# Phase 1: Causal Graph Discovery (PCMCI)
**Goal**: من بيانات train1-4 الطبيعية — اكتشف العلاقات السببية بين الـ sensors  
**Output**: Causal graph → يُستخدم لاحقاً كـ physics constraint للـ Transformer

**Pipeline**:
```
train1-4 (normal only)
    → select non-constant sensors
    → subsample (every 10s for speed)
    → PCMCI (ParCorr, τ_max=5)
    → causal graph (edges + weights)
    → save graph for Transformer training
```

## Cell 1 — Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, pickle
from pathlib import Path

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

print("✓ All imports OK")

✓ All imports OK


## Cell 2 — Load & Clean Data
نحمّل train1-4 (بيانات طبيعية فقط)، نحذف الـ constant columns، ونوحّد الـ features عبر الملفات الأربعة.

In [2]:
DATA_DIR = Path("../data/processed")
META_COLS = {"timestamp", "attack", "label", "attack_p1", "attack_p2", "attack_p3"}
STD_THRESHOLD = 1e-6

def load_normal(file_num: int) -> pd.DataFrame:
    """Load a train file, keep only normal rows (attack==0)."""
    df = pd.read_csv(DATA_DIR / f"train{file_num}.csv")
    if "attack" in df.columns:
        df = df[df["attack"] == 0].reset_index(drop=True)
    return df

# Load all 4 train files
dfs = [load_normal(i) for i in range(1, 5)]
print("Rows per file:", [len(d) for d in dfs])

# Sensor columns = numeric, non-meta
def sensor_cols(df):
    return [c for c in df.columns if c not in META_COLS and df[c].dtype != object]

# Find columns that are non-constant in ALL 4 files
all_sensor = set(sensor_cols(dfs[0]))
for df in dfs[1:]:
    all_sensor &= set(sensor_cols(df))

# Keep only non-constant across all files
non_const = []
for col in sorted(all_sensor):
    if all(df[col].std() > STD_THRESHOLD for df in dfs):
        non_const.append(col)

print(f"\nShared sensor columns  : {len(all_sensor)}")
print(f"Non-constant sensors   : {len(non_const)}")
print(f"Sample sensors         : {non_const[:8]}")

Rows per file: [280799, 291599, 125999, 197999]

Shared sensor columns  : 311
Non-constant sensors   : 129
Sample sensors         : ['1001.13-OUT', '1001.14-OUT', '1001.15-OUT', '1001.16-OUT', '1001.17-OUT', '1001.20-OUT', '1001.5-OUT', '1002.20-OUT']


## Cell 3 — Subsample & Normalize
PCMCI بيكون بطيء على 280K row × 133 sensor.  
نأخذ كل 10 ثواني (subsample=10) → ~90K row, وهذا كافي لاكتشاف العلاقات السببية.

In [3]:
SUBSAMPLE = 10   # take every 10th second — change to 1 for full data (slow)

# Concatenate all 4 files → one long time-series
combined = pd.concat([df[non_const] for df in dfs], ignore_index=True)

# Subsample
data_raw = combined.iloc[::SUBSAMPLE].values.astype(np.float32)

# Z-score normalize (column-wise)
mean = data_raw.mean(axis=0)
std  = data_raw.std(axis=0)
std[std < 1e-8] = 1.0          # avoid div-by-zero on near-constants
data_norm = (data_raw - mean) / std

print(f"Data shape after subsample : {data_norm.shape}")
print(f"  → {data_norm.shape[0]:,} timesteps × {data_norm.shape[1]} sensors")
print(f"  → At τ_max=5, each PCMCI test uses up to {data_norm.shape[0]-5:,} samples")

Data shape after subsample : (89640, 129)
  → 89,640 timesteps × 129 sensors
  → At τ_max=5, each PCMCI test uses up to 89,635 samples


## Cell 4 — Run PCMCI
**الخوارزمية**: PCMCI (Runge et al. 2019)  
**Test**: ParCorr (partial correlation) — سريع، مناسب للـ linear relationships  
**τ_max = 5** → يبحث عن تأثيرات تأخر 1-5 ثواني (بعد الـ subsample = 10-50 ثانية حقيقية)  
**α = 0.05** → مستوى الثقة الإحصائية

In [ ]:
TAU_MAX = 5
ALPHA   = 0.05

# Build tigramite dataframe
dataframe = pp.DataFrame(
    data       = data_norm,
    var_names  = non_const,
    missing_flag = np.nan
)

# Run PCMCI
pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=1)
results = pcmci.run_pcmci(tau_max=TAU_MAX, pc_alpha=ALPHA)

print("\n✓ PCMCI done!")


##
## Step 1: PC1 algorithm for selecting lagged conditions
##

Parameters:
independence test = par_corr
tau_min = 1
tau_max = 5
pc_alpha = [0.05]
max_conds_dim = None
max_combinations = 1




c:\Users\ahmma\anaconda3\envs\digital_twin\lib\site-packages\tigramite\independence_tests\parcorr.py:146: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val, _ = stats.pearsonr(x_vals, y_vals)


## Cell 5 — Extract Significant Edges
نستخرج كل علاقة p-value < 0.05، ونرتبها حسب قوة التأثير (MCI coefficient).

In [ ]:
p_matrix   = results["p_matrix"]    # shape: (N, N, tau_max+1)
val_matrix = results["val_matrix"]  # shape: (N, N, tau_max+1)
N          = len(non_const)

edges = []
for j in range(N):         # target sensor
    for i in range(N):     # source sensor
        for tau in range(1, TAU_MAX + 1):   # tau=0 = instantaneous, skip
            p   = p_matrix[i, j, tau]
            val = val_matrix[i, j, tau]
            if p < ALPHA:
                edges.append({
                    "source"     : non_const[i],
                    "target"     : non_const[j],
                    "lag"        : tau,
                    "coeff"      : round(float(val), 4),
                    "p_value"    : round(float(p),   6),
                })

edges_df = pd.DataFrame(edges).sort_values("coeff", key=abs, ascending=False)
print(f"Significant causal edges found: {len(edges_df)}")
print(f"Unique source sensors         : {edges_df['source'].nunique()}")
print(f"Unique target sensors         : {edges_df['target'].nunique()}")
print("\nTop 15 strongest edges:")
edges_df.head(15)

## Cell 6 — Visualize: Top Edges Heatmap
شو أقوى العلاقات؟ نرسم heatmap للـ sensors اللي عندها أكثر causal links.

In [ ]:
# Top 25 most-connected sensors (by number of edges in or out)
from collections import Counter

all_nodes = list(edges_df["source"]) + list(edges_df["target"])
top_nodes = [n for n, _ in Counter(all_nodes).most_common(25)]

# Build adjacency matrix for top nodes (max |coeff| across lags)
idx = {n: i for i, n in enumerate(top_nodes)}
adj = np.zeros((len(top_nodes), len(top_nodes)))

for _, row in edges_df.iterrows():
    if row["source"] in idx and row["target"] in idx:
        i, j = idx[row["source"]], idx[row["target"]]
        if abs(row["coeff"]) > abs(adj[i, j]):
            adj[i, j] = row["coeff"]

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(adj, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(top_nodes))); ax.set_xticklabels(top_nodes, rotation=90, fontsize=7)
ax.set_yticks(range(len(top_nodes))); ax.set_yticklabels(top_nodes, fontsize=7)
ax.set_title(f"Causal Graph — Top 25 Sensors\n(color = max MCI coefficient across τ=1..{TAU_MAX})", fontsize=11)
plt.colorbar(im, ax=ax, label="MCI coefficient (strength)")
plt.tight_layout()
plt.savefig("../outputs/plots/causal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/plots/causal_heatmap.png")

## Cell 7 — Visualize: Parents per Sensor
لكل sensor — كم عدد الـ parents (المتغيرات اللي تسببه)؟  
هذا يعطيك فكرة عن تعقيد الـ SCM equations لاحقاً.

In [ ]:
parents_count = edges_df.groupby("target")["source"].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(parents_count)), parents_count.values, color="#2196F3", alpha=0.8)
ax.set_xticks(range(len(parents_count)))
ax.set_xticklabels(parents_count.index, rotation=90, fontsize=6)
ax.set_xlabel("Sensor (target)")
ax.set_ylabel("Number of causal parents")
ax.set_title(f"Causal Parents per Sensor  (total edges = {len(edges_df)})")
plt.tight_layout()
plt.savefig("../outputs/plots/causal_parents_count.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSensors with most parents (top 10):")
print(parents_count.head(10).to_string())

## Cell 8 — Save Graph
نحفظ الـ graph بكل تنسيقاته عشان نستعمله لاحقاً في:
- **Causal Loss** أثناء تدريب الـ Transformer
- **Causal Validator** بعد الـ inference

In [ ]:
OUT_DIR = Path("../outputs/causal_graph")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. edges CSV — human readable
edges_df.to_csv(OUT_DIR / "edges.csv", index=False)

# 2. parents dict — {sensor: [(parent_sensor, lag, coeff), ...]}
#    This is the format the Transformer causal loss will consume
parents_dict = {}
for _, row in edges_df.iterrows():
    target = row["target"]
    if target not in parents_dict:
        parents_dict[target] = []
    parents_dict[target].append({
        "parent" : row["source"],
        "lag"    : int(row["lag"]),
        "coeff"  : row["coeff"],
    })

with open(OUT_DIR / "parents.json", "w") as f:
    json.dump(parents_dict, f, indent=2)

# 3. val_matrix + p_matrix numpy arrays (for tigramite plots later)
np.save(OUT_DIR / "val_matrix.npy", val_matrix)
np.save(OUT_DIR / "p_matrix.npy",   p_matrix)

# 4. sensor names list
with open(OUT_DIR / "sensor_names.json", "w") as f:
    json.dump(non_const, f, indent=2)

print("Saved to outputs/causal_graph/:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")

print(f"\nSummary:")
print(f"  Sensors in graph : {len(non_const)}")
print(f"  Causal edges     : {len(edges_df)}")
print(f"  Sensors with parents: {len(parents_dict)}")